This file is to experiment with setting thresholds for our data with NLP-produced similarity scores
- for now, just using some sample file(s) from get_scores()
- sectioning off data by each ..05 of similarity scores
- then, get cohen's kappas on that for comparison

In [1]:
import pandas as pd
from NLP_Eval_for_DE import scores, data
import ast
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix


Welcome to NLP eval for DE, version 1.0.0


In [39]:
testable_data = data.get_testable_data("Example\\inputs\\case study 1 input\\pain points full.csv")
codes = data.get_codes("Example\\inputs\\case study 1 input\\short titles+descriptions.csv")
all_scores = scores.get_BART_scores(testable_data, codes)[1]

#split the lots of scores in the "Similarity scores" column into separate columns
all_scores_expanded = all_scores.copy()
all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]] = pd.DataFrame(all_scores_expanded["Similarity scores"].tolist(), index=all_scores_expanded.index)
# add the "Consensus code" column of testable_data to all_scores_expanded
all_scores_expanded["Consensus code"] = testable_data["Consensus code"].tolist()
#delete the "Similarity scores" column
all_scores_expanded = all_scores_expanded.drop(columns=["Similarity scores"])
all_scores_expanded

Device set to use cuda:0


,Input phrase,1,2,3,4,5,6,7,8,9,10,11,Consensus code
0,cutting wood,0.982985,0.910655,0.729666,0.259546,0.656981,0.579609,0.761801,0.893477,0.173065,0.991660,0.728036,0
1,didn�t know how to use lathe,0.980424,0.860755,0.620371,0.153389,0.575175,0.375660,0.597491,0.791475,0.013808,0.959897,0.853349,1
2,Finding drill,0.989142,0.861889,0.641704,0.369796,0.851760,0.512452,0.708157,0.812235,0.664798,0.992831,0.719243,9
3,Taking out trash,0.983473,0.911427,0.583318,0.339085,0.515239,0.920115,0.777397,0.927079,0.090017,0.971462,0.721145,10
4,Finding clamp,0.994024,0.883549,0.769487,0.503607,0.978408,0.457249,0.842106,0.845430,0.178215,0.993626,0.739827,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,Incorrect size gloves,0.985648,0.948719,0.813139,0.325794,0.674094,0.372528,0.791143,0.795850,0.009630,0.981104,0.853469,3
395,Uncleaned machines from previous users,0.963188,0.694975,0.651299,0.330314,0.863798,0.540830,0.689715,0.503695,0.023085,0.973605,0.883093,6
396,Machine incorrectly set up by previous user,0.990381,0.654063,0.800697,0.343904,0.562126,0.581418,0.570045,0.626751,0.001467,0.981085,0.934683,1
397,Unusable wood scarps were discarded in wrong p...,0.959431,0.922780,0.596967,0.286623,0.944991,0.726632,0.656878,0.741919,0.004700,0.958911,0.616901,8


In [40]:
def eval_filtered(all_scores_expanded, min, max):
    # now filter based on threshold 
    # drop any rows where the highest score from the 11 scores is not between min and max
    all_scores_expanded_filtered = all_scores_expanded[
        (all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].max(axis=1) >= min) 
        & (all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].max(axis=1) <= max)]

    # now, get cohens kappa score for the filtered data
    ground_truths = all_scores_expanded_filtered["Consensus code"].tolist()
    predictions = all_scores_expanded_filtered[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].idxmax(axis=1).tolist()
    # convert predictions from strings to ints
    predictions = [int(x) for x in predictions]
    f1s = f1_score(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11], average=None, zero_division=0.0) 
    mtx = confusion_matrix(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11])
    kappa = cohen_kappa_score(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11], weights=None, sample_weight=None)
    # get average f1 (will see later if this is ok)
    f1 = sum(f1s) / len(f1s)
    return [mtx, f1, kappa]

In [41]:
rows = []
for i, j in [(0.95, 1.0), (0.9, 0.95), (0.85, 0.9), (0.8, 0.85), (0.75, 0.8), (0.7, 0.75), (0.65, 0.7), (0.6, 0.65), (0.55, 0.6), (0.5, 0.55), (0.45, 0.5), (0.4, 0.45), (0.35, 0.4), (0.3, 0.35), (0.25, 0.3), (0.2, 0.25), (0.15, 0.2), (0.1, 0.15), (0.05, 0.1), (0.0, 0.05)]:
    results = eval_filtered(all_scores_expanded, i, j)
    rows.append({"min": i, "max": j, "kappa": results[2], "f1": results[1]})
thresholded_kappas = pd.DataFrame(rows)
thresholded_kappas.to_csv("thresholding_results\\thrKappas_BART_title+desc_painpoints.csv", index=False)
thresholded_kappas

C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(sum0, sum1) / np.sum(sum0)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:745: RuntimeWarning: invalid value encountered in divide
  expected = np.outer(su

,min,max,kappa,f1
0,0.95,1.00,0.141955,0.20692
1,0.90,0.95,-0.113636,0.00000
2,0.85,0.90,NaN,0.00000
3,0.80,0.85,NaN,0.00000
4,0.75,0.80,NaN,0.00000
5,0.70,0.75,NaN,0.00000
6,0.65,0.70,NaN,0.00000
7,0.60,0.65,NaN,0.00000
8,0.55,0.60,NaN,0.00000
9,0.50,0.55,NaN,0.00000
